# Encontro 8 — Logging, Observabilidade e Métricas

Este notebook apresenta práticas de logging em agentes de IA, tracing de ponta a ponta e monitoramento de custos, latência e taxa de erros.

## Objetivos
- Estratégias de logging para agentes de IA (estruturado, níveis, correlação).
- Tracing E2E: cada passo, chamada de LLM, e uso de ferramenta.
- Monitoramento de custos, latência e taxa de erros (KPIs operacionais).

## Dependências (opcional)
Se precisar, instale libs auxiliares para observabilidade:

```bash
pip install python-dotenv rich langsmith opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp
```

Observação: LangSmith requer `LANGCHAIN_API_KEY`, `LANGCHAIN_ENDPOINT`, `LANGCHAIN_TRACING_V2="true"` e opcional `LANGCHAIN_PROJECT`.

In [ ]:
# Configuração básica de logging estruturado
import logging, time, os
from typing import Any, Dict

class SimpleFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        base = {
            'level': record.levelname,
            'ts': int(time.time() * 1000),
            'msg': record.getMessage(),
            'module': record.module,
        }
        # Extra fields (correlation, request_id, etc.)
        if hasattr(record, 'extra') and isinstance(record.extra, dict):
            base.update(record.extra)
        return str(base)

logger = logging.getLogger('observabilidade')
handler = logging.StreamHandler()
handler.setFormatter(SimpleFormatter())
logger.setLevel(logging.INFO)
logger.addHandler(handler)

logger.info('Logging inicializado', extra={'extra': {'component': 'setup'}})


## Tracing E2E com LangChain + callbacks
Exemplo de instrumentação de passos do chain/LLM com callbacks de LangChain. Para LangSmith, garanta que as variáveis de ambiente estão configuradas.

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

class TraceHandler(BaseCallbackHandler):
    def __init__(self, logger):
        self.logger = logger
    def on_chain_start(self, serialized, inputs, **kwargs):
        self.logger.info('chain_start', extra={'extra': {'inputs': inputs}})
    def on_chain_end(self, outputs, **kwargs):
        self.logger.info('chain_end', extra={'extra': {'outputs': outputs}})
    def on_llm_start(self, serialized, prompts, **kwargs):
        self.logger.info('llm_start', extra={'extra': {'prompts': prompts}})
    def on_llm_end(self, response, **kwargs):
        # response contém o texto; token usage pode não estar disponível com todos provedores
        self.logger.info('llm_end', extra={'extra': {'response_preview': str(response)[:200]}})
    def on_llm_error(self, error, **kwargs):
        self.logger.error('llm_error', extra={'extra': {'error': str(error)}})

# Prompt simples e LLM com Gemini (requer GOOGLE_API_KEY)
prompt = PromptTemplate.from_template("Responda com precisão: {question}")
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
chain = prompt | llm | StrOutputParser()

handler = TraceHandler(logger)

# Execução com tracing via callbacks
question = 'Quais estratégias de logging são recomendadas para agentes de IA?'
try:
    resp = chain.invoke({'question': question}, config={'callbacks': [handler]})
    print(resp)
except Exception as e:
    logger.error('exec_error', extra={'extra': {'error': str(e)}})


## Métricas: latência, erros e custos
Capturamos timestamps, contadores de erro e estimativas simples de custo/tokens. Em produção, integrar com Prometheus/Grafana ou LangSmith para dashboards.

In [ ]:
import time
from dataclasses import dataclass, field

@dataclass
class Metrics:
    requests: int = 0
    errors: int = 0
    latencies_ms: list = field(default_factory=list)
    total_cost_usd: float = 0.0

    def avg_latency(self):
        return sum(self.latencies_ms) / len(self.latencies_ms) if self.latencies_ms else 0.0

metrics = Metrics()

def estimate_cost(prompt_len: int, response_len: int) -> float:
    # Placeholder de custo; substitua por tarifação real do provedor
    return 0.000001 * (prompt_len + response_len)

def observed_call(question: str) -> str:
    metrics.requests += 1
    t0 = time.time()
    try:
        out = chain.invoke({'question': question})
        dt = (time.time() - t0) * 1000
        metrics.latencies_ms.append(dt)
        c = estimate_cost(len(question), len(out))
        metrics.total_cost_usd += c
        logger.info('metrics', extra={'extra': {'latency_ms': dt, 'cost_usd': round(c, 6)}})
        return out
    except Exception as e:
        metrics.errors += 1
        logger.error('call_error', extra={'extra': {'error': str(e)}})
        raise

ans = observed_call('Liste três práticas de observabilidade para agentes.')
print(ans)
print({'requests': metrics.requests, 'errors': metrics.errors, 'avg_latency_ms': round(metrics.avg_latency(), 2), 'total_cost_usd': round(metrics.total_cost_usd, 6)})


## Integração com LangSmith (opcional)
Defina as variáveis e execute chamadas para visualizar traces, custos e métricas no LangSmith.

- `LANGCHAIN_API_KEY`
- `LANGCHAIN_ENDPOINT` (ex.: `https://api.smith.langchain.com`)
- `LANGCHAIN_TRACING_V2="true"`
- `LANGCHAIN_PROJECT` (opcional)

Após configurar, re-execute células com o chain para gerar traces.

## Próximos passos
- Adicionar exportação de métricas para Prometheus (via `prometheus_client`).
- Enviar traces para OTLP (OpenTelemetry) e visualizar em Grafana Tempo/Jaeger.
- Padronizar correlação (request_id, session_id) nos logs e traces.